# Kissing Number Problem (d = 11) — Analysis & Verification

The kissing number problem asks: how many non-overlapping unit spheres can simultaneously touch a central unit sphere in $d$ dimensions?

For $d = 11$, the previous best lower bound was **593** (AlphaEvolve, 2025). We construct a configuration of **594** unit spheres with zero overlap, establishing a new lower bound.

In [1]:
import json
import itertools
from decimal import Decimal, getcontext

getcontext().prec = 80

ZERO = Decimal(0)
TWO = Decimal(2)
FOUR = Decimal(4)

## 1. Verification function

The verifier from the [Einstein Arena](https://einsteinarena.com) `kissing-number-d11` problem. Uses 80-digit decimal arithmetic for exact checking.

In [2]:
def _to_dec(x):
    return Decimal(str(x))


def _exact_check(vectors):
    d = len(vectors[0])
    dec_vecs = [[_to_dec(x) for x in vec] for vec in vectors]

    squared_norms = [sum(x * x for x in vec) for vec in dec_vecs]
    if min(squared_norms) == ZERO:
        return False
    max_sq_norm = max(squared_norms)

    min_sq_dist = None
    for p, q in itertools.combinations(dec_vecs, 2):
        sq_dist = sum((a - b) ** 2 for a, b in zip(p, q))
        if min_sq_dist is None or sq_dist < min_sq_dist:
            min_sq_dist = sq_dist

    return min_sq_dist >= max_sq_norm


def _overlap_loss(vectors):
    d = len(vectors[0])
    scaled = []
    for vec in vectors:
        norm_sq = sum((_to_dec(x) ** 2 for x in vec), ZERO)
        if norm_sq == ZERO:
            raise ValueError("All vectors must be non-zero")
        norm = norm_sq.sqrt()
        scaled.append([(_to_dec(x) * TWO) / norm for x in vec])

    n = len(scaled)
    total = ZERO
    for i in range(n):
        for j in range(i + 1, n):
            sq = sum(((scaled[i][k] - scaled[j][k]) ** 2 for k in range(d)), ZERO)
            if sq < FOUR:
                total += (TWO - sq.sqrt())
    return float(total)


def evaluate(data: dict) -> float:
    vectors = data["vectors"]
    if len(vectors) != 594 or len(vectors[0]) != 11:
        raise ValueError(f"Expected shape (594, 11), got ({len(vectors)}, {len(vectors[0])})")
    if _exact_check(vectors):
        print("Exact check passed — valid kissing configuration!")
        return 0.0
    return _overlap_loss(vectors)

## 2. Load solution

In [3]:
with open("solutions/ours_2026.json") as f:
    ours = json.load(f)

print(f"Number of vectors: {len(ours['vectors'])}")
print(f"Dimension:         {len(ours['vectors'][0])}")

Number of vectors: 594
Dimension:         11


## 3. Verify score

A score of exactly **0.0** confirms a valid kissing configuration — no overlaps.

In [4]:
score = evaluate(ours)
print(f"Score (overlap loss): {score}")
print()
if score == 0.0:
    print("The configuration is VALID.")
    print("This proves the kissing number in dimension 11 is at least 594.")
else:
    print(f"Overlap remains: {score}")

Exact check passed — valid kissing configuration!
Score (overlap loss): 0.0

The configuration is VALID.
This proves the kissing number in dimension 11 is at least 594.
